# ProverbGap MCQ -- Paper-First Analysis Notebook
**Generated:** 2026-06-22_10-42-02  
**Dataset:** v68 N=5 corpus-fallback baseline (180 MCQs, full blind audit)  
**Purpose:** Produce publication-ready tables/figures and a stratified 60-item human-validation sample without calling any API.  

> WARNING: OpenRouter budget is exhausted; v70 failed after 83 partial MCQs. This notebook uses v68 as the production dataset.


In [ ]:
import os, json, math
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

BASE = 'kaggle_run_logs/v68/openrouter_pilot1_test_output'
OUT_DIR = 'paper_first_outputs_2026-06-22_10-42-02'
os.makedirs(OUT_DIR, exist_ok=True)

print('Output directory:', OUT_DIR)


In [ ]:
# Load v68 generated MCQs and audit results
mcq = pd.read_csv(f'{BASE}/pilot1_test_generated_mcqs.csv')
audit = pd.read_csv(f'{BASE}/pilot1_test_audit_results.csv')

# Merge on shared identifiers
merge_keys = ['mcq_id', 'generator_model', 'variant', 'language', 'correct_label']
df = pd.merge(mcq, audit, on=merge_keys, how='inner')

print('MCQs:', df.shape[0])
print('Columns:', df.columns.tolist()[:12], '...')
print('Generation status:')
print(df['generation_status'].value_counts())


In [ ]:
def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    z = stats.norm.ppf(1 - alpha / 2)
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    margin = z * math.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
    return (max(0.0, centre - margin), min(1.0, centre + margin))

def compute_metrics(sub, prefix=''):
    n = len(sub)
    if n == 0:
        return pd.Series({prefix+'n': 0})
    k_correct = int(sub['consensus_correct'].sum())
    ci_low, ci_high = wilson_ci(k_correct, n)
    return pd.Series({
        prefix+'n': n,
        prefix+'perfect_consensus_rate': (sub['consensus_frac'] == 1.0).mean(),
        prefix+'hcw_rate': ((sub['consensus_frac'] >= 0.75) & (sub['consensus_correct'] == 0)).mean(),
        prefix+'consensus_accuracy': sub['consensus_correct'].mean(),
        prefix+'consensus_accuracy_ci_low': ci_low,
        prefix+'consensus_accuracy_ci_high': ci_high,
        prefix+'partial_or_fallback_rate': sub['generation_status'].isin(['partial','length_fallback','parse_fallback','hard_fallback']).mean(),
        prefix+'hard_fallback_rate': (sub['generation_status'] == 'length_fallback').mean(),
        prefix+'mean_fallback_count': sub['fallback_count'].mean(),
        prefix+'mean_nli_replaced': sub['nli_replaced'].mean(),
        prefix+'mean_leak_replaced': sub['leak_replaced'].mean(),
        prefix+'mean_length_replaced': sub['length_replaced'].mean(),
        prefix+'duplicate_options': int(sub['duplicate_options'].sum()),
    })


In [ ]:
# Aggregate metrics
agg = compute_metrics(df).to_frame().T
agg['correct_key_balance'] = str(df['correct_label'].value_counts().sort_index().to_dict())
agg.to_csv(f'{OUT_DIR}/table1_aggregate_metrics.csv', index=False)

print('=== Table 1: Aggregate metrics ===')
for col in agg.columns:
    if col.startswith('n') or col == 'correct_key_balance':
        print(f'{col:30s}: {agg[col].iloc[0]}')
    else:
        print(f'{col:30s}: {agg[col].iloc[0]:.4f}')


In [ ]:
# Per-language metrics with Wilson 95% CIs
per_lang = df.groupby('language').apply(compute_metrics, include_groups=False).reset_index()
per_lang.to_csv(f'{OUT_DIR}/table2_per_language_metrics.csv', index=False)

print('=== Table 2: Per-language metrics ===')
print(per_lang[['language','n','consensus_accuracy','consensus_accuracy_ci_low','consensus_accuracy_ci_high','hcw_rate','partial_or_fallback_rate']].to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(7,4))
x = np.arange(len(per_lang))
y = per_lang['consensus_accuracy']
err_low = y - per_lang['consensus_accuracy_ci_low']
err_high = per_lang['consensus_accuracy_ci_high'] - y
ax.bar(x, y, yerr=[err_low, err_high], capsize=5, color=['#1f77b4','#ff7f0e','#2ca02c'])
ax.axhline(0.25, color='red', linestyle='--', label='Random (25%)')
ax.set_xticks(x)
ax.set_xticklabels(per_lang['language'])
ax.set_ylabel('Consensus accuracy')
ax.set_ylim(0,1)
ax.set_title('Figure 1: Per-language consensus correctness (v68)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig1_per_language_consensus.png', bbox_inches='tight')
plt.show()


In [ ]:
# Per-variant and per-generator metrics
per_variant = df.groupby('variant').apply(compute_metrics, include_groups=False).reset_index()
per_variant.to_csv(f'{OUT_DIR}/table3_per_variant_metrics.csv', index=False)

per_gen = df.groupby('generator_model').apply(compute_metrics, include_groups=False).reset_index()
per_gen.to_csv(f'{OUT_DIR}/table4_per_generator_metrics.csv', index=False)

print('=== Table 3: Per-variant metrics ===')
print(per_variant[['variant','n','consensus_accuracy','hcw_rate','partial_or_fallback_rate']].to_string(index=False))

print('\n=== Table 4: Per-generator metrics ===')
print(per_gen[['generator_model','n','consensus_accuracy','hcw_rate','partial_or_fallback_rate']].to_string(index=False))


In [ ]:
# Position bias: accuracy by correct-key position
pos = df.groupby('correct_label')['consensus_correct'].agg(['mean','count']).reset_index()
pos.columns = ['correct_label','accuracy','n']
pos.to_csv(f'{OUT_DIR}/table5_position_bias.csv', index=False)

# Chi-square test for position bias
contingency = pd.crosstab(df['correct_label'], df['consensus_correct'])
chi2, pvalue, _, _ = stats.chi2_contingency(contingency)
print('Position bias chi-square: p =', pvalue)

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(pos['correct_label'], pos['accuracy'], color='steelblue')
ax.axhline(df['consensus_correct'].mean(), color='red', linestyle='--', label='Overall accuracy')
ax.set_ylabel('Consensus accuracy')
ax.set_xlabel('Correct-key position')
ax.set_title('Figure 2: Position bias analysis (v68)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig2_position_bias.png', bbox_inches='tight')
plt.show()


In [ ]:
# Generation status distribution and fallback/repair breakdown
status_counts = df['generation_status'].value_counts().reset_index()
status_counts.columns = ['status','count']
status_counts.to_csv(f'{OUT_DIR}/table6_status_distribution.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].bar(status_counts['status'], status_counts['count'], color='coral')
axes[0].set_title('Figure 3a: Generation status distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

repair_means = df[['length_replaced','leak_replaced','nli_replaced']].mean()
axes[1].bar(repair_means.index, repair_means.values, color=['#2ca02c','#ff7f0e','#9467bd'])
axes[1].set_title('Figure 3b: Mean replacements per MCQ')
axes[1].set_ylabel('Mean count')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig3_status_and_repairs.png', bbox_inches='tight')
plt.show()


In [ ]:
# Failure-mode summary (effect sizes from project history; corpus-fallback magnitude updated from v68)
failure_modes = pd.DataFrame({
    'failure_mode': ['Same-family exploitation', 'Chain-of-thought collapse', 'Multi-stage-prompt triviality',
                     'Position bias', 'API/provider fragility', 'Corpus fallback quality collapse'],
    'locus': ['Audit/evaluator', 'Auditor prompting', 'Generator prompting', 'MCQ assembly', 'Infrastructure', 'Fallback sampler'],
    'magnitude_pp': [31.1, 34.7, 100.0, 60.6, 90.0, 43.9],
    'v68_magnitude': [np.nan, np.nan, np.nan, 60.6, np.nan, 43.9]
})
failure_modes.to_csv(f'{OUT_DIR}/table7_failure_taxonomy.csv', index=False)

fig, ax = plt.subplots(figsize=(8,4.5))
ax.barh(failure_modes['failure_mode'], failure_modes['magnitude_pp'], color='firebrick')
ax.set_xlabel('Approximate effect size (percentage points)')
ax.set_title('Figure 4: Failure-mode summary (v58-v68)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig4_failure_modes.png', bbox_inches='tight')
plt.show()


In [ ]:
# Per-variant / per-generator HCW heatmap
heatmap_data = df.pivot_table(values='consensus_correct', index='generator_model', columns='variant', aggfunc='mean')
fig, ax = plt.subplots(figsize=(8,4))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax)
ax.set_title('Figure 5: Consensus accuracy by generator x variant (v68)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig5_generator_variant_accuracy.png', bbox_inches='tight')
plt.show()


In [ ]:
# Generate stratified 60-item human-validation sample
# 20 per language, stratified by generation_status and consensus_correct
samples = []
for lang, group in df.groupby('language'):
    sampled = []
    strata = group.groupby(['generation_status', 'consensus_correct'])
    per_stratum = max(1, 20 // strata.ngroups)
    for _, sub in strata:
        sampled.append(sub.sample(n=min(per_stratum, len(sub)), random_state=42))
    sampled = pd.concat(sampled, ignore_index=True) if sampled else group.iloc[:0]
    if len(sampled) < 20:
        remaining = group[~group['mcq_id'].isin(sampled['mcq_id'])]
        extra = remaining.sample(min(20 - len(sampled), len(remaining)), random_state=42)
        sampled = pd.concat([sampled, extra], ignore_index=True)
    sampled = sampled.sample(n=min(20, len(sampled)), random_state=42).reset_index(drop=True)
    sampled['validation_id'] = [f'{lang}_{i+1}' for i in range(len(sampled))]
    samples.append(sampled)

val_sample = pd.concat(samples, ignore_index=True)
val_sample.to_csv(f'{OUT_DIR}/human_validation_sample_60.csv', index=False)

print('Validation sample size:', val_sample.shape[0])
print('Per language:', val_sample['language'].value_counts().to_dict())
print('Per status:', val_sample['generation_status'].value_counts().to_dict())
print('Per consensus correctness:', val_sample['consensus_correct'].value_counts().to_dict())


## Summary

This notebook produced:
- Aggregate v68 metrics and key-balance check.
- Per-language consensus accuracy with Wilson 95% CIs.
- Per-variant and per-generator tables.
- Position-bias analysis.
- Generation-status and repair-breakdown figures.
- Failure-mode summary figure.
- Generator x variant accuracy heatmap.
- Stratified 60-item human-validation sample.

All artifacts are saved in `paper_first_outputs_2026-06-22_10-42-02/`.

## Next steps
1. Recruit native-speaker annotators for the 60-item validation sample.
2. Draft paper sections using the generated tables/figures.
3. Renew OpenRouter budget before any further Kaggle experiments.
